In [ ]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torch.utils.data import DataLoader

In [2]:
class UNetLite(nn.Module):
    """Ultra-Lite U-Net for 1x32x32 -> 1x32x32 reconstructions.

    Architecture follows the provided table:
    - Enc1: Conv2d 1->8, kernel=3, stride=1, padding=1 (keeps 32x32)
    - Pool1: MaxPool2d 2x2 -> 16x16
    - Bottle: Conv2d 8->16, kernel=3, stride=1, padding=1 (keeps 16x16)
    - Up1: ConvTranspose2d 16->8, kernel=2, stride=2, padding=0 (16x16 -> 32x32)
    - Cat with Enc1 -> 16 channels
    - Dec1: Conv2d 16->8, kernel=3, stride=1, padding=1 (keeps 32x32)
    - Final: Conv2d 8->out_channels, kernel=1
    """

    def __init__(self, in_channels=1, out_channels=1):
        super(UNetLite, self).__init__()
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottle = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )

        # Decoder / Upsample
        self.up1 = nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2, padding=0)
        self.dec1 = nn.Sequential(
            nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )

        # Final 1x1 conv to map to output channels
        self.final = nn.Conv2d(8, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)        # -> [B,8,32,32]
        x_p = self.pool1(x1)     # -> [B,8,16,16]

        # Bottleneck
        xb = self.bottle(x_p)    # -> [B,16,16,16]

        # Upsample
        xu = self.up1(xb)        # -> [B,8,32,32]

        # Concatenate skip connection from encoder (channels: 8 + 8 = 16)
        x_cat = torch.cat([xu, x1], dim=1)  # -> [B,16,32,32]

        # Decoder
        xd = self.dec1(x_cat)    # -> [B,8,32,32]

        # Final linear conv
        out = self.final(xd)     # -> [B,out_channels,32,32]
        return out

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Pad(2),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [4]:
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
from torch.utils.data import Subset

small_trainset = Subset(trainset, range(10000))
trainloader = DataLoader(small_trainset, batch_size=512, shuffle=True)

In [5]:
model = UNetLite().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
print("Starting Training...")
for epoch in range(2):
    running_loss = 0.0
    for i, data in enumerate(trainloader):
        inputs, _ = data
        inputs = inputs.to(device)
        # Implement the Training Step
        # 1. Clear the gradients from the previous iteration
        optimizer.zero_grad()

        # 2. Forward pass: Compute the predicted reconstruction
        outputs = model(inputs)

        # 3. Compute loss, perform backpropagation, and update weights
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()


        running_loss += loss.item()
        if i % 10 == 9:
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] Loss: {running_loss / 100:.4f}')
            running_loss = 0.0

print("Finished Training.")

Starting Training...
[Epoch 1, Batch 10] Loss: 0.0799
[Epoch 1, Batch 20] Loss: 0.0454
[Epoch 2, Batch 10] Loss: 0.0200
[Epoch 2, Batch 20] Loss: 0.0125
Finished Training.


## UNetLite Architecture Table (TBD filled)

| Stage | Layer Type | Input (C x H x W) | Output (C x H x W) | Kernel | Stride | Padding |
|---|---|---:|---:|---:|---:|---:|
| Input | - | 1 x 32 x 32 | 1 x 32 x 32 | - | - | - |
| Enc 1 | Conv2d + ReLU | 1 x 32 x 32 | 8 x 32 x 32 | 3 x 3 | 1 | 1 |
| Pool 1 | MaxPool2d | 8 x 32 x 32 | 8 x 16 x 16 | 2 x 2 | 2 x 2 | 0 |
| Bottle | Conv2d + ReLU | 8 x 16 x 16 | 16 x 16 x 16 | 3 x 3 | 1 | 1 |
| Up 1 | ConvTranspose2d | 16 x 16 x 16 | 8 x 32 x 32 | 2 x 2 | 2 | 0 |
| Cat 1 | Concatenate (Enc 1) | 8 x 32 x 32 (up) + 8 x 32 x 32 (skip) | 16 x 32 x 32 | - | - | - |
| Dec 1 | Conv2d + ReLU | 16 x 32 x 32 | 8 x 32 x 32 | 3 x 3 | 1 | 1 |
| Final | Conv2d (Linear) | 8 x 32 x 32 | 1 x 32 x 32 | 1 x 1 | 1 | 0 |

**Justification (conv formulas):**
- For Conv2d: output = floor((H + 2*padding - kernel)/stride) + 1. Using kernel=3, padding=1, stride=1 keeps spatial dims unchanged (32 -> 32, 16 -> 16).
- For MaxPool2d with kernel=2, stride=2, padding=0 halves spatial dims (32 -> 16).
- For ConvTranspose2d: output = (H-1)*stride - 2*padding + kernel + output_padding. With kernel=2, stride=2, padding=0 -> (16-1)*2 + 2 = 32.

This configuration ensures encoder and decoder feature maps have matching spatial sizes for skip connections.